# <b>Differential Gene Expression Analysis</b>
## Using DeSEQ2 with biological replicates

In [ ]:
### import packages using the pydeseqEnv environment
import custom_functions_pydeseqEnv as cf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import scanpy as sc
import seaborn as sns
import warnings
from custom_functions_pydeseqEnv import plot_volcano_df, plot_volcano_df_html, return_de
from matplotlib.pyplot import rc_context

In [ ]:
# Set the seed
np.random.seed(123)

#warnings.filterwarnings("ignore")

## Loading the data

In [ ]:
adata = sc.read_h5ad("../data/adata_all_samples_preprocessed.h5ad")

In [ ]:
annotate = sc.read_h5ad('../data/adata_all_samples_annotated.h5ad')

In [ ]:
### We'll normalize and log transform the data in adata here
### Deseq will take in data from the "counts" layer which is unnormalized.
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

In [ ]:
### Then we add annotations, clusters, and UMAP coordinates
adata.obs['celltype'] = annotate.obs.celltype.copy() 
adata.obs['celltype_broad'] = annotate.obs.celltype_broad.copy() 
adata.obs['leiden'] = annotate.obs.leiden.copy()
adata.obsm['X_umap'] = annotate.obsm['X_umap'].copy()

In [ ]:
with rc_context({'figure.figsize': (6, 6)}):
    sc.pl.umap(adata, color = ['celltype'], frameon = False, s = 10)

## Using DESeq across high-level comparisons

In [ ]:
### Create comparison groups for each tissue vs the rest of the tissues for Deseq2 analysis
target = "Kidney"
adata.obs['comparison-group-Kidney'] = adata.obs['tissue'].apply(
    lambda x: target if x == target else 'Rest'
)

target = "Mesentery"
adata.obs['comparison-group-Mesentery'] = adata.obs['tissue'].apply(
    lambda x: target if x == target else 'Rest'
)

target = "Thoracic_Aorta"
adata.obs['comparison-group-Thoracic-Aorta'] = adata.obs['tissue'].apply(
    lambda x: target if x == target else 'Rest'
)

In [ ]:
%%time
### High level comparisons across whole-tissue
deseqRes = []

celltype = 'whole_tissue'
group_list = ['comparison-group-Kidney', 'comparison-group-Mesentery', 'comparison-group-Thoracic-Aorta']
group1_list = ['Rest', 'Rest', 'Rest']
group2_list = ['Kidney', 'Mesentery', 'Thoracic-Aorta']

for i in range(len(group_list)):
    print(f'Comparing across {group1_list[i]} mesothelium in {celltype}')
    
    adata_subset = adata.copy()
    celltype = celltype
    group = group_list[i]
    group1 = group1_list[i] 
    group2 = group2_list[i]
    
    ### Get the counts of cells per sample and check if any are less than 3
    sample_counts = adata_subset.obs['sample_id'].value_counts()
    failing_samples = sample_counts[sample_counts < 3]

    ### Check if there are any failing samples before moving on to DEG analysis
    if not failing_samples.empty:
        print("There are less than 3 cells of this cell type in the following samples:")
        print(failing_samples)
        deseqRes
    else:
        ### Run DESeq and save results
        de = return_de(adata, celltype, group, group1, group2)
        deseqRes.append(de)

        ### Generate and save volcano plots
        plot_volcano_df(de, celltype, group, group1, group2, return_fig = False, save = False)
        plot_volcano_df_html(de, celltype, group, group1, group2, return_fig = False, save = False)
        plt.show()

deseqResults1 = pd.concat(deseqRes)
deseqResults1.to_csv('../output/DEGs/Mesothelium_whole_tissue_vs_rest_deseq_.txt', sep = '\t')

In [ ]:
%%time
### High level comparisons across cell types
deseqRes = []

group_list = ['comparison-group-Kidney', 'comparison-group-Mesentery', 'comparison-group-Thoracic-Aorta']
group1_list = ['Rest', 'Rest', 'Rest']
group2_list = ['Kidney', 'Mesentery', 'Thoracic-Aorta']

celltypes = adata.obs.celltype.unique()

for i in range(len(group_list)):
    for celltype in celltypes:
        print(f'Comparing across {group_list[i]} in taPVAT {celltype}')
              
        adata_subset = adata[adata.obs['celltype'] == celltype].copy()
        celltype = celltype
        group = group_list[i]
        group1 = group1_list[i] 
        group2 = group2_list[i]
        
        ### Get the counts of cells per sample and check if any are less than 3
        sample_counts = adata_subset.obs['sample_id'].value_counts()
        failing_samples = sample_counts[sample_counts < 3]

        ### Check if there are any failing samples before moving on to DEG analysis
        if not failing_samples.empty:
            print("There are less than 3 cells of this cell type in the following samples:")
            print(failing_samples)
            deseqRes
        else:
            ### Run DESeq and save results
            de = return_de(adata_subset, celltype, group, group1, group2)
            deseqRes.append(de)

            ### Generate and save volcano plots
            plot_volcano_df(de, celltype, group, group1, group2, return_fig = False, save = True)
            plot_volcano_df_html(de, celltype, group, group1, group2, return_fig = False, save = True)
            plt.show()
        
deseqResults2 = pd.concat(deseqRes)
deseqResults2.to_csv('../output/DEGs/Mesothelium_vs_Rest_celltype_deseq_all_genes_all.txt', sep = '\t')

## Saving all DESeq results as a single tsv file

In [ ]:
deseqResults1 = pd.read_csv('../output/DEGs/Mesothelium_whole_tissue_vs_rest_deseq_.txt', sep = '\t', index_col = 0)
deseqResults2 = pd.read_csv('../output/DEGs/Mesothelium_vs_Rest_celltype_deseq_all_genes_all.txt', sep = '\t', index_col = 0)

In [ ]:
deseqResults_list = [deseqResults1, deseqResults2]
deseqResults_all = pd.concat(deseqResults_list)

In [ ]:
deseqResults_all

In [ ]:
deseqResults_all.to_csv('../output/DEGs/Mesothelium_all_comparisons_vs_rest_deseq_all_genes_all.txt', sep = '\t')

In [ ]:
deseq_degs_all = deseqResults_all[deseqResults_all.padj <= 0.05]

In [ ]:
deseq_degs_all = deseqResults_all[(deseqResults_all.padj <= 0.05) & ((deseqResults_all.log2FoldChange >= 1)|(deseqResults_all.log2FoldChange <= -1))]\
    .sort_values('stat', ascending = False)

In [ ]:
deseq_degs_all.to_csv('../output/DEGs/Mesothelium_all_comparisons_vs_rest_deseq_all_degs_pvalue_0.05.txt', sep = '\t')

In [ ]:
top_genes = deseq_degs_all[deseq_degs_all['log2FoldChange'] >= 1]
bottom_genes = deseq_degs_all[deseq_degs_all['log2FoldChange'] <= -1]

In [ ]:
top_genes.to_csv('../output/DEGs/Mesothelium_all_comparisons_vs_rest_deseq_all_degs_pvalue_0.05_top.txt', sep = '\t')
bottom_genes.to_csv('../output/DEGs/Mesothelium_all_comparisons_vs_rest_deseq_all_degs_pvalue_0.05_bottom.txt', sep = '\t')

In [ ]:
bottom_genes